In [3]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

RUTA_YAML = Path("../../data/Modulator.yaml")

# =============================================================================
# 2. CREACIÓN DE LA "BASE DE DATOS CORRECTA" (Extracción en vivo)
# =============================================================================
print("📡 1. Extrayendo datos puros de los archivos .bin para crear la BBDD...")

materiales = ['carton', 'cristal', 'plastico']
condiciones = {'frio': 15.0, 'templado': 40.0, 'caliente': 90.0}
muestras = ['1', '2']

variables_clave = ['doppler_variance_energy', 'dH_dt_mean', 'svd_sigma_ratio']
datos_limpios = []

for material in materiales:
    ruta_base = Path(f"../../data/{material}")
    for cond_str, temp_val in condiciones.items():
        for m in muestras:
            tx_path = ruta_base / cond_str / f"iq_tx_{m}.bin"
            rx_path = ruta_base / cond_str / f"iq_rx_{m}.bin"
            
            if not tx_path.exists() or not rx_path.exists():
                continue
                
            try:
                H = compute_channel_matrix_from_iq_paths(
                    tx_path=tx_path, rx_path=rx_path, yaml_path=RUTA_YAML,
                    fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
                    output_order="mk", verbose=False
                )
                features = extract_channel_matrix_features(H, input_order="mk")
                
                # Guardamos solo lo que nos importa
                fila = {var: features[var] for var in variables_clave if var in features}
                fila['temperature'] = temp_val
                fila['material'] = material
                fila['muestra'] = f"{cond_str.capitalize()} - Muestra {m}"
                
                datos_limpios.append(fila)
            except Exception as e:
                print(f"⚠️ Error procesando {material}/{cond_str}/{m}: {e}")

df_total = pd.DataFrame(datos_limpios)
print(f"✅ BBDD Creada: {len(df_total)} muestras perfectas listas para IA.\n")

# =============================================================================
# 3. CONFIGURACIÓN DEL SVR Y VALIDACIÓN CRUZADA
# =============================================================================
# Elige qué material quieres que la IA adivine (los otros 2 se usarán para estudiar)
MATERIAL_TEST = 'carton'  

print("=" * 50)
print(f"🧠 2. Entrenando SVR (Aprendiendo de {(set(materiales) - {MATERIAL_TEST})})")
print(f"🎯 3. Testeando a ciegas a través de: {MATERIAL_TEST.upper()}")
print("=" * 50)

# Separamos Train y Test
df_train = df_total[df_total['material'] != MATERIAL_TEST]
df_test = df_total[df_total['material'] == MATERIAL_TEST]

if df_test.empty:
    print(f"❌ Error: No se encontraron datos para el material de test '{MATERIAL_TEST}'.")
else:
    X_train_bruto = df_train[variables_clave]
    y_train = df_train['temperature']
    
    X_test_bruto = df_test[variables_clave]
    y_test = df_test['temperature']
    nombres_test = df_test['muestra']

    # Escalador vital para el SVR
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_bruto)
    X_test_scaled = scaler.transform(X_test_bruto)

    # Cerebro SVR (C alto para ajustarse bien a los pocos datos limpios que tenemos)
    modelo_svr = SVR(kernel='rbf', C=500.0, epsilon=0.5, gamma='scale')
    modelo_svr.fit(X_train_scaled, y_train)

    # =============================================================================
    # 4. RESULTADOS FINALES
    # =============================================================================
    predicciones = modelo_svr.predict(X_test_scaled)

    print("\n📊 RESULTADOS DE LA PREDICCIÓN:\n")
    for nombre, temp_real, temp_pred in zip(nombres_test, y_test, predicciones):
        error = abs(temp_real - temp_pred)
        print(f"🌡️ {nombre.ljust(22)} -> Real: {temp_real:4.1f} ºC | IA: {temp_pred:4.1f} ºC | Error: {error:4.1f} ºC")

    print("-" * 50)
    mae = mean_absolute_error(y_test, predicciones)
    # Protegemos el R2 si hay muy pocos datos
    if len(y_test) > 1: 
        r2 = r2_score(y_test, predicciones)
        print(f"📈 Puntuación R²: {r2:.2f} (1.0 es la perfección)")
    print(f"📉 Error Medio Absoluto (MAE): {mae:.2f} ºC")

📡 1. Extrayendo datos puros de los archivos .bin para crear la BBDD...
✅ BBDD Creada: 18 muestras perfectas listas para IA.

🧠 2. Entrenando SVR (Aprendiendo de {'cristal', 'plastico'})
🎯 3. Testeando a ciegas a través de: CARTON

📊 RESULTADOS DE LA PREDICCIÓN:

🌡️ Frio - Muestra 1       -> Real: 15.0 ºC | IA: 28.3 ºC | Error: 13.3 ºC
🌡️ Frio - Muestra 2       -> Real: 15.0 ºC | IA: 35.7 ºC | Error: 20.7 ºC
🌡️ Templado - Muestra 1   -> Real: 40.0 ºC | IA: 48.5 ºC | Error:  8.5 ºC
🌡️ Templado - Muestra 2   -> Real: 40.0 ºC | IA: 47.4 ºC | Error:  7.4 ºC
🌡️ Caliente - Muestra 1   -> Real: 90.0 ºC | IA: 70.5 ºC | Error: 19.5 ºC
🌡️ Caliente - Muestra 2   -> Real: 90.0 ºC | IA: 71.7 ºC | Error: 18.3 ºC
--------------------------------------------------
📈 Puntuación R²: 0.75 (1.0 es la perfección)
📉 Error Medio Absoluto (MAE): 14.62 ºC


In [5]:
import sys
import pandas as pd
from pathlib import Path
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

RUTA_YAML = Path("../../data/Modulator.yaml")
variables_clave = ['doppler_variance_energy', 'dH_dt_mean', 'svd_sigma_ratio']

# =============================================================================
# 2. CARGAR Y FUSIONAR BBDD ANTIGUAS (ENTRENAMIENTO)
# =============================================================================
print("🧠 1. Cargando y fusionando BBDD 1 y BBDD 2 (Entrenamiento)...")

ruta_bbdd_1 = '../dataset_features_temperatura.csv'
ruta_bbdd_2 = '../../data/dataset_features_temperatura.csv'

try:
    df_1 = pd.read_csv(ruta_bbdd_1)
    df_2 = pd.read_csv(ruta_bbdd_2)
    # Unimos ambas bases de datos masivas
    df_train = pd.concat([df_1, df_2], ignore_index=True)
    # Limpiamos por si hay algún hueco vacío en las variables que nos interesan
    df_train = df_train.dropna(subset=variables_clave + ['temperature'])
    print(f"✅ BBDD Histórica lista: {len(df_train)} muestras cargadas para entrenar.")
except Exception as e:
    print(f"❌ Error al cargar los CSV antiguos: {e}")
    exit()

X_train_bruto = df_train[variables_clave]
y_train = df_train['temperature']

# =============================================================================
# 3. EXTRAER DATOS EN VIVO PARA TEST (TESTEO)
# =============================================================================
print("\n📡 2. Extrayendo archivos .bin para usarlos como Test Ciego...")

materiales = ['carton', 'cristal', 'plastico']
condiciones = {'frio': 15.0, 'templado': 40.0, 'caliente': 90.0}
muestras = ['1', '2']

datos_test = []

for material in materiales:
    ruta_base = Path(f"../../data/{material}")
    for cond_str, temp_val in condiciones.items():
        for m in muestras:
            tx_path = ruta_base / cond_str / f"iq_tx_{m}.bin"
            rx_path = ruta_base / cond_str / f"iq_rx_{m}.bin"
            
            if not tx_path.exists() or not rx_path.exists():
                continue
                
            try:
                H = compute_channel_matrix_from_iq_paths(
                    tx_path=tx_path, rx_path=rx_path, yaml_path=RUTA_YAML,
                    fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
                    output_order="mk", verbose=False
                )
                features = extract_channel_matrix_features(H, input_order="mk")
                
                fila = {var: features[var] for var in variables_clave if var in features}
                fila['temperature'] = temp_val
                fila['nombre'] = f"[{material.upper()}] {cond_str.capitalize()} - M. {m}"
                
                datos_test.append(fila)
            except Exception:
                pass # Ignoramos errores individuales para mantener la consola limpia

df_test = pd.DataFrame(datos_test)
X_test_bruto = df_test[variables_clave]
y_test = df_test['temperature']
nombres_test = df_test['nombre']

print(f"✅ Test listo: {len(df_test)} muestras extraídas.\n")

# =============================================================================
# 4. CONFIGURACIÓN DEL SVR (ESCALADO + ENTRENAMIENTO)
# =============================================================================
print("=" * 60)
print("⚙️ 3. Entrenando SVR (Esto puede tardar unos segundos por el volumen de datos...)")
print("=" * 60)

# Escalador Z-Score (Crítico porque las BBDD antiguas tienen números gigantescos)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bruto)
X_test_scaled = scaler.transform(X_test_bruto)

# Configuramos SVR. Ponemos un C moderado para que no intente memorizar el ruido extremo
modelo_svr = SVR(kernel='rbf', C=100.0, epsilon=2.0, gamma='scale')
modelo_svr.fit(X_train_scaled, y_train)

# =============================================================================
# 5. PREDICCIÓN Y RESULTADOS FINALES
# =============================================================================
predicciones = modelo_svr.predict(X_test_scaled)

print("\n🎯 RESULTADOS FINALES DE LA PREDICCIÓN (ENTRENADO CON HISTÓRICO):\n")

for nombre, temp_real, temp_pred in zip(nombres_test, y_test, predicciones):
    error = abs(temp_real - temp_pred)
    print(f"🌡️ {nombre.ljust(30)} -> Real: {temp_real:4.1f} ºC | IA: {temp_pred:4.1f} ºC | Error: {error:4.1f} ºC")

print("-" * 60)
mae = mean_absolute_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print(f"📉 Error Medio Absoluto (MAE): {mae:.2f} ºC")
print(f"📈 Puntuación R²: {r2:.2f} (1.0 es la perfección)")
print("=" * 60)

🧠 1. Cargando y fusionando BBDD 1 y BBDD 2 (Entrenamiento)...
✅ BBDD Histórica lista: 58238 muestras cargadas para entrenar.

📡 2. Extrayendo archivos .bin para usarlos como Test Ciego...
✅ Test listo: 18 muestras extraídas.

⚙️ 3. Entrenando SVR (Esto puede tardar unos segundos por el volumen de datos...)

🎯 RESULTADOS FINALES DE LA PREDICCIÓN (ENTRENADO CON HISTÓRICO):

🌡️ [CARTON] Frio - M. 1           -> Real: 15.0 ºC | IA: 32.3 ºC | Error: 17.3 ºC
🌡️ [CARTON] Frio - M. 2           -> Real: 15.0 ºC | IA: 32.9 ºC | Error: 17.9 ºC
🌡️ [CARTON] Templado - M. 1       -> Real: 40.0 ºC | IA: 29.3 ºC | Error: 10.7 ºC
🌡️ [CARTON] Templado - M. 2       -> Real: 40.0 ºC | IA: 27.1 ºC | Error: 12.9 ºC
🌡️ [CARTON] Caliente - M. 1       -> Real: 90.0 ºC | IA: -54.7 ºC | Error: 144.7 ºC
🌡️ [CARTON] Caliente - M. 2       -> Real: 90.0 ºC | IA: -26.9 ºC | Error: 116.9 ºC
🌡️ [CRISTAL] Frio - M. 1          -> Real: 15.0 ºC | IA: -1.3 ºC | Error: 16.3 ºC
🌡️ [CRISTAL] Frio - M. 2          -> Real: 15.0